## Environment Setup

In [2]:
from google.colab import drive
import os
import json

drive.mount('/content/drive')

DATA_PATH = '/content/drive/MyDrive/DV-TM/DATA'
print('Files found:', os.listdir(DATA_PATH))

def load_jsonl(path):
    with open(path, 'r') as f:
        return [json.loads(line) for line in f]

train_cleaned = load_jsonl(os.path.join(DATA_PATH, 'train_cleaned.jsonl'))
test_retokenized = load_jsonl(os.path.join(DATA_PATH, 'test_retokenized.jsonl'))

print(f'Train cleaned: {len(train_cleaned)} samples')
print(f'Test retokenized: {len(test_retokenized)} samples')

Mounted at /content/drive
Files found: ['test.jsonl', 'train.jsonl', 'train_cleaned.jsonl', 'test_retokenized.jsonl']
Train cleaned: 3142 samples
Test retokenized: 800 samples


## **Data Preprocessing**

To ensure a rigorous evaluation of the model's performance and to prevent overfitting, we performed a stochastic partitioning of the `train_cleaned` dataset. We implemented a 70/15/15 split, allocating 70% of the samples for the training phase, 15% for hyperparameter tuning (validation set), and reserving the final 15% as an internal test set for unbiased performance estimation. Crucially, we fixed the random seed to 42; this ensures the reproducibility of the experiment, allowing for consistent comparisons across different neural architectures by ensuring that the model always encounters the same data points in each subset.

In [3]:
import random

random.seed(42)

indices = list(range(len(train_cleaned)))
random.shuffle(indices)

n = len(train_cleaned)
n_val  = int(n * 0.15)
n_test = n_val
n_train = n - n_val - n_test

train_data = [train_cleaned[i] for i in indices[:n_train]]
val_data   = [train_cleaned[i] for i in indices[n_train:n_train + n_val]]
test_data  = [train_cleaned[i] for i in indices[n_train + n_val:n_train + n_val + n_test]]

print(f"Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")

Train: 2200 | Val: 471 | Test: 471


The execution confirms the exact distribution of the samples according to the defined split. This partitioning allows the model to be optimized on the training portion while providing nearly 1,000 unseen examples for the validation and testing phases. This rigorous isolation is essential for detecting whether the Neural Network is identifying generalized linguistic features or if it is over-optimizing to the specific syntactic templates present in the training set.

## **Data Partitioning Strategy and Ground Truth Necessity**

The decision to partition the `train_cleaned` dataset into three distinct subsets (Training, Validation, and Internal Test) was driven by an essential technical requirement identified during the initial data exploration phase.

**Primary Technical Rationale: Absence of Labels in the Official Test Set**

Upon analysis of the `test.jsonl` (and its processed version `test_retokenized.jsonl`) provided for this project, it was observed that the `tags` field is missing. Since Named Entity Recognition (NER) is a **supervised learning** task, calculating quantitative performance metrics—such as **Precision, Recall, and F1-score**—is mathematically impossible without the **Ground Truth** (the actual labels).
Consequently, we extracted a labeled portion from the training data to create a **"Gold" Internal Test Set**, which serves as the only reliable benchmark for the model's performance evaluation.

**Role of the Official Unlabeled Test Set**

As a consequence of the data constraints identified in the previous section, the official test set provided by the professor (`test.jsonl` and its retokenized version) serves a distinct purpose in our experimental pipeline. Since this dataset lacks ground-truth labels, it cannot be utilized for quantitative benchmarking or statistical validation. Instead, it is reserved exclusively for **Qualitative Analysis** and **Inference Testing**.

By processing this unlabeled data, we can observe how the **Neural Network** performs "in the wild," manually inspecting its ability to assign IOB2 tags to raw, unseen job postings. This step is crucial for verifying the practical utility of the model beyond mere metrics, allowing us to evaluate the consistency of its predictions and its behavior when faced with the specific linguistic structures chosen by the instructor for the final evaluation.

## **Feature Extraction and Token-Label Alignment**


In [4]:
# Extraction for the Training Set
# Separating the input features (tokens) from the target variables (labels)
train_tokens = [item["tokens"] for item in train_data]
train_labels = [item["labels"] for item in train_data]

# Extraction for the Validation Set
val_tokens = [item["tokens"] for item in val_data]
val_labels = [item["labels"] for item in val_data]

# Extraction for the Internal Test Set (derived from the initial split)
test_tokens = [item["tokens"] for item in test_data]
test_labels = [item["labels"] for item in test_data]

# --- Data Verification ---
print(f"Number of sentences in Train: {len(train_tokens)}")
print(f"Number of sentences in Val: {len(val_tokens)}")
print(f"Number of sentences in Test: {len(test_tokens)}\n")

# Displaying the first sample of the training set to verify structural alignment
print("--- EXAMPLE [Train Index 0] ---")
print("Tokens :", train_tokens[0])
print("Labels :", train_labels[0])

Number of sentences in Train: 2200
Number of sentences in Val: 471
Number of sentences in Test: 471

--- EXAMPLE [Train Index 0] ---
Tokens : ['SmartTech', '(', 'San', 'Francisco', ')', 'seeks', 'DevOps', 'Engineer', '.', 'Must', 'know', 'project', 'management', '.']
Labels : ['B-COMPANY', 'O', 'B-LOCATION', 'I-LOCATION', 'O', 'O', 'B-JOBTITLE', 'I-JOBTITLE', 'O', 'O', 'O', 'B-SKILL', 'I-SKILL', 'O']


Following the dataset partitioning, the next crucial step is the isolation of input features from their target variables. In Named Entity Recognition (NER), the inputs are sequences of words (tokens), and the targets are parallel sequences of categorical tags (labels). We extracted these paired lists for the training, validation, and internal test sets.

A critical requirement for sequence labeling architectures, such as BiLSTMs, is the strict positional alignment between the input token and its corresponding label. As verified through the data inspection step, the tokenization aligns perfectly with the IOB2 format. For instance, multi-word entities are correctly segmented and labeled sequentially (e.g., the token `"San"` maps to `"B-LOCATION"`, while `"Francisco"` accurately maps to `"I-LOCATION"`). This structural integrity guarantees that the Neural Network receives unambiguous supervisory signals during the backpropagation phase.


## **Text Vectorization and Integer Encoding**

In [5]:
from tensorflow.keras.preprocessing.text import Tokenizer

# 1. Initialize the tokenizer.
# lower=False: We preserve the original case (e.g., "Apple" vs "apple") as capitalization
#              is a strong morphological feature for Named Entity Recognition.
# oov_token="<OOV>": A dedicated token to safely handle new, unseen words in Val and Test sets.
word_tokenizer = Tokenizer(lower=False, oov_token="<OOV>")

# 2. Build the vocabulary EXCLUSIVELY on the training data to prevent Data Leakage.
word_tokenizer.fit_on_texts(train_tokens)

# 3. Convert textual tokens into sequences of integers (Integer Encoding).
X_train = word_tokenizer.texts_to_sequences(train_tokens)
X_val   = word_tokenizer.texts_to_sequences(val_tokens)
X_test  = word_tokenizer.texts_to_sequences(test_tokens)

# Calculate the vocabulary size (required later for the Embedding layer).
# We add +1 because index 0 is strictly reserved by Keras for padding purposes.
vocab_size = len(word_tokenizer.word_index) + 1

print(f"Vocabulary size: {vocab_size}")
print("\n--- ENCODED TEXT EXAMPLE [Train Index 0] ---")
print("Original:", train_tokens[0])
print("Numeric :", X_train[0])

Vocabulary size: 103

--- ENCODED TEXT EXAMPLE [Train Index 0] ---
Original: ['SmartTech', '(', 'San', 'Francisco', ')', 'seeks', 'DevOps', 'Engineer', '.', 'Must', 'know', 'project', 'management', '.']
Numeric : [92, 41, 31, 32, 42, 43, 52, 10, 2, 9, 44, 86, 87, 2]


To process textual data through a Neural Network, the input tokens must be projected into a numerical space. We achieved this via Integer Encoding using the Keras `Tokenizer`. Two critical hyperparameters were explicitly configured during initialization: first, `lower=False` ensures that case sensitivity is preserved, which is a highly informative feature for identifying entities like Companies or Locations; second, `oov_token="<OOV>"` establishes a robust fallback mechanism for any previously unseen terms encountered during validation or testing.

Crucially, the `fit_on_texts` method was applied strictly to the training corpus. This isolates the learning environment, preventing *Data Leakage* and ensuring that the model's vocabulary perfectly reflects a real-world scenario where the network operates with a "Closed Vocabulary" of known terms. The tokens were subsequently converted into sequences of integers, setting the foundational input structure for the upcoming Embedding layer.

### **Output Commentary**

> **Execution Result Analysis:**
> `Vocabulary size: 103`
>
> The calculated vocabulary size explicitly exposes the structural limitations of the synthetic dataset. With only 103 unique tokens (including the `<OOV>` token and the reserved `0` index for padding), the model operates within a severely constrained lexical environment. While this guarantees rapid convergence and high performance on identically structured test data, it severely compromises the model's generalization capabilities in open-domain scenarios.
>
> **Sequence Mapping Verification:**
> `Original: ['SmartTech', '(', 'San', 'Francisco', ...]`
> `Numeric : [92, 41, 31, 32, ...]`
>
> The sample output confirms the flawless deterministic 1:1 mapping between strings and integers. We can observe the system's consistency: words that appear frequently or share syntactic roles receive specific identifiers, and identical tokens are mapped accurately (e.g., the period `'.'` at the end of the sentence is assigned the integer `2`, mapping identically wherever it appears).


## **Vocabulary Inspection and Dataset Profiling**

In [6]:
# Sorting the vocabulary by index to inspect all unique words learned by the tokenizer
sorted_vocabulary = sorted(word_tokenizer.word_index.items(), key=lambda item: item[1])

print(f"The vocabulary contains {len(sorted_vocabulary)} unique words:\n")
for word, index in sorted_vocabulary:
    print(f"{index}: {word}")

The vocabulary contains 102 unique words:

1: <OOV>
2: .
3: :
4: in
5: ,
6: at
7: a
8: Manager
9: Must
10: Engineer
11: position
12: with
13: Required
14: Location
15: Analyst
16: TechCorp
17: Seattle
18: Marketing
19: JavaScript
20: communication
21: Excellent
22: opportunity
23: Skills
24: Sales
25: Representative
26: InnovateLabs
27: data
28: analysis
29: New
30: York
31: San
32: Francisco
33: needs
34: proficient
35: HR
36: Specialist
37: Austin
38: PrimeTech
39: FutureSoft
40: Denver
41: (
42: )
43: seeks
44: know
45: Agile
46: CloudServices
47: role
48: Key
49: skill
50: DataSystems
51: Inc
52: DevOps
53: Seeking
54: UX
55: Designer
56: located
57: have
58: skills
59: Atlanta
60: Apply
61: now
62: machine
63: learning
64: We
65: 're
66: looking
67: for
68: to
69: join
70: Software
71: cloud
72: computing
73: Digital
74: Ventures
75: available
76: Requirements
77: Business
78: is
79: hiring
80: experience
81: Product
82: Global
83: Solutions
84: Boston
85: SQL
86: project
87: mana

In [7]:
# Extracting and displaying the 20 most frequent words in the training set
# This helps identify the underlying templates used to generate the dataset
most_common = sorted(word_tokenizer.word_counts.items(), key=lambda x: x[1], reverse=True)

print("\n--- Top 20 Most Frequent Words ---")
for word, count in most_common[:20]:
    print(f"'{word}' appears {count} times")


--- Top 20 Most Frequent Words ---
'.' appears 3927 times
':' appears 1975 times
'in' appears 1112 times
',' appears 898 times
'at' appears 893 times
'a' appears 636 times
'Manager' appears 460 times
'Must' appears 441 times
'Engineer' appears 436 times
'position' appears 432 times
'with' appears 430 times
'Required' appears 425 times
'Location' appears 419 times
'Analyst' appears 417 times
'TechCorp' appears 265 times
'Seattle' appears 253 times
'Marketing' appears 249 times
'JavaScript' appears 243 times
'communication' appears 240 times
'Excellent' appears 239 times


To empirically investigate the lexical diversity of the training data, we conducted a direct inspection of the tokenizer's internal dictionary and word frequency distribution. The analysis exposed a severely constrained lexicon of exactly 103 unique tokens.

Furthermore, extracting the top 20 most frequent words revealed a corpus heavily dominated by punctuation (periods appearing 3,927 times, colons 1,975 times) and structural prepositions (e.g., `'in'` appearing 1,112 times, `'at'` 893 times). Among the few semantic words, generic terms like `'Manager'` (460 occurrences), `'Engineer'` (436), and `'position'` (432) heavily outweigh specific entities. For instance, only a single company name (`'TechCorp'`, 265 times) and a single location (`'Seattle'`, 253 times) manage to enter the top 20 list.



This stark lack of vocabulary variance mathematically confirms the synthetic nature of the dataset. Rather than representing natural, open-domain human language—which typically follows a Zipfian distribution with thousands of rare words—this data is generated through rigid, repetitive templates centered around a few anchor words (like "in" or "at"). This structural bottleneck makes the model highly susceptible to overfitting on specific syntactic patterns, directly motivating the necessity to later introduce pre-trained contextual knowledge (GloVe) to achieve real-world generalization.

## **Custom Target Variable Encoding and Label Space Definition**

In [8]:
# 1. Extract all unique tags present in the training set
all_tags = set(tag for doc in train_labels for tag in doc)

# Add a special tag for padding purposes
all_tags.add("_PAD_")

# 2. Create the mapping dictionaries
# We sort the tags to ensure the exact same index association across different runs
tag2idx = {tag: idx for idx, tag in enumerate(sorted(all_tags))}
idx2tag = {idx: tag for tag, idx in tag2idx.items()}

# Calculate the total number of distinct classes (tags)
num_tags = len(tag2idx)

# 3. Helper function to convert lists of textual labels into integers
def encode_labels(labels_list, tag_dict):
    return [[tag_dict[tag] for tag in doc] for doc in labels_list]

# Apply the mapping to the datasets
Y_train = encode_labels(train_labels, tag2idx)
Y_val   = encode_labels(val_labels, tag2idx)
Y_test  = encode_labels(test_labels, tag2idx)

print(f"Total unique Tags (including padding): {num_tags}")
print("Tag -> Index Dictionary:\n", tag2idx)
print("\n--- ENCODED LABELS EXAMPLE [Train Index 0] ---")
print("Original:", train_labels[0])
print("Numeric :", Y_train[0])

Total unique Tags (including padding): 10
Tag -> Index Dictionary:
 {'B-COMPANY': 0, 'B-JOBTITLE': 1, 'B-LOCATION': 2, 'B-SKILL': 3, 'I-COMPANY': 4, 'I-JOBTITLE': 5, 'I-LOCATION': 6, 'I-SKILL': 7, 'O': 8, '_PAD_': 9}

--- ENCODED LABELS EXAMPLE [Train Index 0] ---
Original: ['B-COMPANY', 'O', 'B-LOCATION', 'I-LOCATION', 'O', 'O', 'B-JOBTITLE', 'I-JOBTITLE', 'O', 'O', 'O', 'B-SKILL', 'I-SKILL', 'O']
Numeric : [0, 8, 2, 6, 8, 8, 1, 5, 8, 8, 8, 3, 7, 8]


Unlike the input textual tokens, which were processed using the standard Keras `Tokenizer`, the target IOB2 labels required a custom encoding pipeline. Standard NLP tokenizers automatically apply text-cleaning heuristics (such as lowercasing and punctuation stripping) which would destructively alter the strict categorical structure of tags like `B-COMPANY`. To preserve this structural integrity, we implemented a deterministic custom dictionary mapping.

First, the exhaustive set of unique entity tags was extracted exclusively from the training set. A designated `_PAD_` class was then intentionally injected into this set. This is a critical engineering step: since neural networks require fixed-length arrays, shorter sequences must be artificially padded. By assigning a distinct, isolated class to these padding tokens, we ensure the TimeDistributed classification layer does not misinterpret empty spaces as actual Named Entities. Finally, the classes were alphabetically sorted to guarantee exact reproducibility across different execution environments, and all label sequences were mapped to their corresponding integer representations.

---

### **Output Commentary**

> **Execution Result Analysis:**
> `Total unique Tags (including padding): 10`
>
> The extraction process successfully identified the 9 standard labels dictated by the dataset's IOB2 schema (combining the 'Beginning' and 'Inside' prefixes for Company, Job Title, Location, and Skill, plus the 'Outside' tag 'O'). With the addition of the specialized `_PAD_` token, the total output dimensionality of our neural network is definitively set to 10 classes.
>
> **Dictionary and Sequence Verification:**
> The generated `tag2idx` dictionary confirms a clean, alphabetical integer mapping. Due to ASCII sorting rules, standard alphabetical tags occupy indices `0` through `8`, while the special `_PAD_` token is safely isolated at index `9`.
>
> Looking at the encoded sequence for `[Train Index 0]`, the structural alignment is perfectly preserved post-encoding. The entity `'B-COMPANY'` is correctly projected to `0`, the background tag `'O'` to `8`, and multi-word entities maintain their sequential logic (e.g., `'B-LOCATION'` and `'I-LOCATION'` mapped to `2` and `6`). This exact, bijective, 1:1 mapping between input tokens and output classes is the fundamental prerequisite for training the BiLSTM architecture.

## **Sequence Padding and Dimensional Standardization**

In [9]:
from keras.preprocessing.sequence import pad_sequences

# 1. Calculate the maximum sentence length within our Training Set
max_len = max([len(seq) for seq in X_train])
print(f"Maximum sequence length: {max_len}")

# 2. Padding for the Input Features (X)
# We use value=0 (the default) to fill the empty temporal steps for the tokens
X_train_pad = pad_sequences(X_train, maxlen=max_len, padding='post', value=0)
X_val_pad   = pad_sequences(X_val, maxlen=max_len, padding='post', value=0)
X_test_pad  = pad_sequences(X_test, maxlen=max_len, padding='post', value=0)

# 3. Padding for the Target Labels (Y)
# Crucial step: we use the specific index of our "_PAD_" tag as the fill value
pad_tag_value = tag2idx["_PAD_"]

Y_train_pad = pad_sequences(Y_train, maxlen=max_len, padding='post', value=pad_tag_value)
Y_val_pad   = pad_sequences(Y_val, maxlen=max_len, padding='post', value=pad_tag_value)
Y_test_pad  = pad_sequences(Y_test, maxlen=max_len, padding='post', value=pad_tag_value)

print("\n--- DATA DIMENSIONS READY FOR THE NETWORK ---")
print(f"Shape X_train: {X_train_pad.shape} | Y_train: {Y_train_pad.shape}")
print(f"Shape X_val  : {X_val_pad.shape}   | Y_val  : {Y_val_pad.shape}")
print(f"Shape X_test : {X_test_pad.shape}  | Y_test  : {Y_test_pad.shape}")

print("\n--- PADDED EXAMPLE [Train Index 0] ---")
print("X (Text)  :", X_train_pad[0])
print("Y (Label) :", Y_train_pad[0])

Maximum sequence length: 18

--- DATA DIMENSIONS READY FOR THE NETWORK ---
Shape X_train: (2200, 18) | Y_train: (2200, 18)
Shape X_val  : (471, 18)   | Y_val  : (471, 18)
Shape X_test : (471, 18)  | Y_test  : (471, 18)

--- PADDED EXAMPLE [Train Index 0] ---
X (Text)  : [92 41 31 32 42 43 52 10  2  9 44 86 87  2  0  0  0  0]
Y (Label) : [0 8 2 6 8 8 1 5 8 8 8 3 7 8 9 9 9 9]


Neural networks inherently require fixed-size input tensors to process data efficiently in parallel batches. Since natural language sentences intrinsically vary in length, we applied a sequence padding strategy. We dynamically computed the maximum sequence length (`max_len`) present in the training corpus and standardized all subsets (Training, Validation, and Internal Test) to this uniform dimension by appending artificial tokens at the end of shorter sentences (`padding='post'`).

For the input features (X), the empty temporal steps were filled using the standard index `0`. Crucially, for the target sequences (Y), we padded the arrays using the specific integer index previously assigned to our custom `_PAD_` class. This meticulous alignment guarantees that the network's TimeDistributed classification layer does not erroneously compute loss gradients over empty padding spaces, preserving the mathematical integrity of the Named Entity Recognition task.

### **Output Commentary**

> **Execution Result Analysis:**
> `Maximum sequence length: 18`
>
> The calculated maximum length of 18 tokens provides further empirical evidence regarding the synthetic simplicity of the dataset. Real-world job postings typically consist of complex, multi-clause paragraphs spanning dozens or hundreds of words. A maximum threshold of only 18 tokens indicates that the model will be optimized on highly truncated, simplistic linguistic structures. This serves as a critical baseline limitation that will be explicitly addressed later in our Stress Test, where we demonstrate the necessity of expanding the context window (`max_len=50`) to process real-world documents.
>
> **Dimensional Verification:**
> `Shape X_train: (2200, 18) | Y_train: (2200, 18)`
>
> The tensor shapes definitively confirm that our data is now perfectly formatted for our initial neural network architecture. Both inputs (X) and targets (Y) share the exact same temporal dimensionality across all 2,200 training samples.
>
> **Padding Alignment:**
> `X (Text)  : [92 41 ... 0  0  0  0]`
> `Y (Label) : [ 0  8 ... 9  9  9  9]`
>
> The padded example vividly illustrates the success of our custom label encoding. In the input sequence (X), the final four steps are populated with standard Keras `0` padding. In perfect positional synchrony, the target sequence (Y) is padded with `9` (the precise index of our `_PAD_` class). This structural alignment ensures that the network will safely ignore these final four steps during the learning process. With the data rigorously pre-processed, aligned, and padded, we are now structurally prepared to construct our first model: a standard, unidirectional LSTM. This initial architecture will serve as a foundational baseline to evaluate sequence learning capabilities before attempting to capture more complex, bidirectional linguistic contexts.

## **Transfer Learning via Pre-trained Word Embeddings (GloVe)**

In [12]:
import os
import numpy as np

# 1. Define the Google Drive path for the embeddings directory
EMBEDDINGS_PATH = '/content/drive/MyDrive/DV-TM/embeddings'
glove_embedding_path = os.path.join(EMBEDDINGS_PATH, 'glove.6B.100d.txt')
embedding_dim = 100

print("Loading GloVe vectors from Google Drive into memory...")
embeddings_index = {}

# 2. Parsing the Stanford GloVe text file directly from Drive
if not os.path.exists(glove_embedding_path):
    raise FileNotFoundError(f"Missing GloVe file at {glove_embedding_path}. Ensure Drive is mounted.")

with open(glove_embedding_path, 'r', encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coefs

print(f"Successfully loaded {len(embeddings_index)} word vectors.")

# 3. Constructing our custom embedding matrix for the Neural Network
glove_matrix = np.zeros((vocab_size, embedding_dim))
hits = 0
misses = 0

for word, i in word_tokenizer.word_index.items():
    # CRUCIAL MODIFICATION: We look up the lowercased version of the word in GloVe!
    embedding_vector = embeddings_index.get(word.lower())

    if embedding_vector is not None:
        glove_matrix[i] = embedding_vector
        hits += 1
    else:
        # Printing missing words to understand semantic gaps
        print(f"Not found in GloVe: {word}")
        misses += 1

print(f"\nMatrix completed! Words found in GloVe (hits): {hits} | Words not found (misses): {misses}")

Loading GloVe vectors from Google Drive into memory...
Successfully loaded 400000 word vectors.
Not found in GloVe: <OOV>
Not found in GloVe: TechCorp
Not found in GloVe: InnovateLabs
Not found in GloVe: PrimeTech
Not found in GloVe: FutureSoft
Not found in GloVe: CloudServices
Not found in GloVe: DataSystems
Not found in GloVe: DevOps
Not found in GloVe: SmartTech

Matrix completed! Words found in GloVe (hits): 93 | Words not found (misses): 9


# **MAGARI AGGIUNGERE SEZIONE PER SPIEGARE GLOVE**

As observed during the vocabulary inspection, our training corpus is syntactically rigid and severely constrained. Training an Embedding layer entirely from scratch on such a limited dataset would inevitably result in poor generalization capabilities. To bridge this semantic knowledge gap, we implemented Transfer Learning using Stanford’s Pre-trained **GloVe (Global Vectors for Word Representation)**. Specifically, we utilized the 100-dimensional vectors trained on the massive Wikipedia 2014 + Gigaword 5 corpus.

To optimize our computational pipeline within the Google Colab environment, the 400,000 pre-computed embedding vectors were persistently hosted on a connected Google Drive infrastructure (`/content/drive/MyDrive/DV-TM/embeddings`). This architectural choice bypasses redundant, bandwidth-heavy downloads during session initializations and allows the Python runtime to parse the semantic matrix directly into memory, significantly accelerating the experimental workflow.

**The Casing Resolution Strategy**

A significant engineering challenge arises when mapping our customized dataset to the GloVe dictionary. In the preprocessing phase, we intentionally configured our Keras `Tokenizer` with `lower=False` to preserve capitalization, as capital letters serve as a fundamental morphological signal for Named Entity Recognition (e.g., distinguishing "Apple" the company from "apple" the fruit). However, the GloVe 6B corpus is exclusively lowercased.

To resolve this conflict without sacrificing our case-sensitive input tracking, we engineered a custom mapping bridge: we iterate through our case-preserved `word_index`, but we query the `embeddings_index` dictionary using `word.lower()`. This architectural trick allows us to retrieve the rich, pre-trained semantic vector of the word while simultaneously retaining its original case-sensitive integer ID for the neural network's input layer.

**Output Commentary**

> The local extraction process successfully mapped 91% of our vocabulary (93 out of 102 unique tokens) into the GloVe semantic space. However, closely analyzing the 9 specific "misses" (the out-of-vocabulary terms) provides crucial insights into the synthetic nature of the dataset:
>
> 1.  **The Fallback Token:** `<OOV>` is naturally missing, as it is an artificial padding/fallback tag created by Keras rather than a real linguistic word.
> 2.  **Synthetic Corporate Entities:** Words like `TechCorp`, `InnovateLabs`, `PrimeTech`, `FutureSoft`, `CloudServices`, `DataSystems`, and `SmartTech` are entirely fictional company names generated exclusively for this dataset. Consequently, they do not exist in the real-world Wikipedia/Gigaword corpus used to train the GloVe algorithm.
>
> **Architectural Impact:** > Because we initialized our `glove_matrix` with `np.zeros`, the embedding vectors for these 9 missing words remain entirely blank (arrays of zeroes). This means the Neural Network will receive no prior semantic knowledge for these specific entities. Instead, it will be forced to identify them purely based on **contextual syntax** (e.g., recognizing that a capitalized word immediately following the preposition "at" is likely a company, regardless of its intrinsic semantic meaning). This exact constraint flawlessly justifies the introduction of sequence modeling architectures, leading us to the construction of our first baseline model: the Unidirectional LSTM.

## **Baseline Unidirectional LSTM Architecture Construction**


In [13]:
from keras.models import Sequential
from keras.layers import Input, Embedding, LSTM, Dense, TimeDistributed

# Architectural hyperparameters
LSTM_UNITS = 64
EMBEDDING_DIM = 100

# Initialize a sequential neural network
model_lstm_glove = Sequential()

# 1. Input Layer
# Keras 3 standard: explicitly declaring the input tensor shape upfront.
# 'max_len' defines the sequence length (18 temporal steps).
model_lstm_glove.add(Input(shape=(max_len,)))

# 2. Embedding Layer (Knowledge Transfer)
model_lstm_glove.add(Embedding(
    input_dim=vocab_size,          # 103 unique tokens
    output_dim=EMBEDDING_DIM,      # 100-dimensional GloVe vectors
    weights=[glove_matrix],        # Injecting our custom pre-trained matrix
    trainable=False,               # Freezing weights to prevent catastrophic forgetting
    mask_zero=True                 # Critical: instructs the network to ignore the padding zeros
))

# 3. Recurrent Layer
# return_sequences=True ensures the LSTM outputs a prediction for EACH temporal step
# rather than just a single vector at the end of the sentence.
model_lstm_glove.add(LSTM(units=LSTM_UNITS, return_sequences=True))

# 4. Output Classification Layer
# TimeDistributed applies the Dense layer to every single temporal step independently.
# Softmax outputs a probability distribution across our 10 possible label classes.
model_lstm_glove.add(TimeDistributed(Dense(units=num_tags, activation='softmax')))

# 5. Model Compilation
# Using 'sparse_categorical_crossentropy' because our target labels are integers,
# not one-hot encoded vectors.
model_lstm_glove.compile(optimizer='adam',
                         loss='sparse_categorical_crossentropy',
                         metrics=['accuracy'])

# Display the architectural summary
model_lstm_glove.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 18, 100)        │        10,300 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 18, 64)         │        42,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 18, 10)         │           650 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 53,190 (207.77 KB)

 Trainable params: 42,890 (167.54 KB)

 Non-trainable params: 10,300 (40.23 KB)

To establish a foundational baseline for sequence learning, we constructed a Unidirectional Long Short-Term Memory (LSTM) network. The architecture is instantiated as a sequential stack of four distinct layers, each addressing a specific requirement of the Named Entity Recognition task.

The topology begins with an explicit **Input Layer** parameterized to our temporal constraint (`max_len=18`). This tensor is fed into the **Embedding Layer**, which serves as the semantic projection space. Crucially, we injected our pre-computed `glove_matrix` as static weights and set `trainable=False`. This "freezing" prevents the backward propagation of errors from catastrophically altering the highly optimized GloVe representations during the initial training phases. Furthermore, we enabled `mask_zero=True`, dynamically instructing the network to halt gradient calculations when encountering the padding sequences, thereby maintaining computational efficiency and preventing noise accumulation.

The core sequential reasoning is handled by the **LSTM Layer** comprising 64 hidden units. By setting `return_sequences=True`, we force the recurrent layer to emit a hidden state vector at every temporal step, preserving the spatial layout of the sentence. Finally, the **TimeDistributed Output Layer** utilizes a Dense projection with a Softmax activation function. The `TimeDistributed` wrapper is an architectural necessity for many-to-many sequence tasks: it ensures that the Dense layer independently evaluates the hidden state of the LSTM at each time step, computing a distinct probability distribution across our 10 defined IOB2 categorical classes. The network is optimized via the Adam algorithm, utilizing Sparse Categorical Crossentropy to natively process our integer-encoded target variables.

---

### **Output Commentary**

> **Execution Result Analysis: Architectural Summary**
>
> The Keras `model.summary()` provides a highly transparent verification of our network's tensor transformations and parameter distribution.
>
> **1. Tensor Dimensionality Flow:**
> The summary validates the structural integrity of our `return_sequences=True` and `TimeDistributed` configuration. As the data flows through the network, the temporal dimension of `18` is flawlessly preserved across all layers. The input text is projected from a discrete index sequence into continuous vectors `(None, 18, 100)`, processed contextually into a hidden representation `(None, 18, 64)`, and finally projected into the categorical output space `(None, 18, 10)`, explicitly matching our `num_tags`.
>
> **2. Parameter Distribution Analysis:**
> * **Non-trainable params (10,300):** This mathematically confirms the successful freezing of the Embedding layer. The number perfectly aligns with our vocabulary shape ($103 \text{ tokens} \times 100 \text{ dimensions}$). These GloVe weights will remain entirely static during training.
> * **LSTM parameters (42,240):** This encapsulates the recurrent kernel, standard weights, and biases associated with the LSTM's complex internal gating mechanisms (Forget, Input, Output gates). These are the primary parameters responsible for learning the syntactical context of the job descriptions.
> * **Dense parameters (650):** The final projection layer accounts for mapping the 64 LSTM units to the 10 categorical outputs ($64 \times 10 \text{ weights} + 10 \text{ biases}$).
>
> The total trainable parameter count of **42,890** represents a highly constrained, lightweight model. This explicitly sets our baseline: we are now testing if ~42K parameters are sufficient to generalize the underlying patterns of this synthetic dataset by processing sentences strictly from left to right.

## **Model Training**

In [14]:
# Set the number of epochs and the batch size
EPOCHS = 10
BATCH_SIZE = 32

print("Starting training of the LSTM model with GloVe...")
history_lstm_glove = model_lstm_glove.fit(
    X_train_pad,
    Y_train_pad,
    validation_data=(X_val_pad, Y_val_pad),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)
print("Training completed!")

Starting training of the LSTM model with GloVe...
Epoch 1/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 7s 42ms/step - accuracy: 0.6336 - loss: 1.2096 - val_accuracy: 0.8616 - val_loss: 0.5545
Epoch 2/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.9578 - loss: 0.2693 - val_accuracy: 0.9917 - val_loss: 0.1186
Epoch 3/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.9975 - loss: 0.0896 - val_accuracy: 0.9981 - val_loss: 0.0634
Epoch 4/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.9985 - loss: 0.0580 - val_accuracy: 0.9981 - val_loss: 0.0467
Epoch 5/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.9985 - loss: 0.0458 - val_accuracy: 0.9981 - val_loss: 0.0384
Epoch 6/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 0.9985 - loss: 0.0387 - val_accuracy: 0.9981 - val_loss: 0.0331
Epoch 7/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.9985 - loss: 0.0335 - val_accuracy: 0.9981 - val_loss: 0.0289
Epoch 8/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - accuracy

The Unidirectional LSTM model was trained over 10 epochs using a batch size of 32. The optimization was driven by the Adam algorithm, which dynamically adapts the learning rate during gradient descent, paired with the Sparse Categorical Crossentropy loss function to evaluate the discrete integer-encoded predictions. To rigorously monitor the generalization capability of the architecture and prevent overfitting, the isolated validation set (`X_val_pad`, `Y_val_pad`) was evaluated at the conclusion of every epoch. The training process was executed successfully, logging both token-level accuracy and loss trajectory across the temporal sequence.

---
### **Output Commentary: Training Dynamics**

> **Execution Result Analysis:**
>
> The training logs demonstrate an exceptionally rapid convergence. During the initial epoch, the model quickly established a solid foundation, achieving a training accuracy of 63.36% and a validation accuracy of 86.16%. By the third epoch, the performance effectively plateaued at an optimal level, reaching 99.85% training accuracy and 99.81% validation accuracy.
>
> Throughout the remaining epochs (4 through 10), the accuracy metrics remained remarkably stable. However, the model continued to refine its predictive confidence. This is explicitly evidenced by the loss metrics: the validation loss consistently decreased from 0.0467 at Epoch 4 down to a minimal 0.0192 by Epoch 10.
>
> **Overfitting Evaluation:**
> Crucially, the validation loss strictly followed the downward trajectory of the training loss without any signs of divergence or spiking. This clean mathematical alignment confirms that the Unidirectional LSTM successfully mapped the target distribution and generalized well to the validation set, showing absolutely no evidence of overfitting during the 10-epoch training cycle.